# Model Deployment (Pandas)
## End to end notebook

In [1]:
# Initialisation
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
# Tweaking function for housing dataset
def tweak_housing(df):
    df = df.copy()
    df['zipcode'] = df['zipcode'].astype(str).astype('category')
    df['date'] = pd.to_datetime(dict(year=df['date_year'],
                                     month=df['date_month'],
                                     day=df['date_day']))
    # Replace 0 with np.nan (not pd.NA)
    df['yr_renovated'] = df['yr_renovated'].replace(0, np.nan)
    return df[['id', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
               'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above',
               'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
               'sqft_living15', 'sqft_lot15', 'date']]

In [3]:
# Make the pipeline
# --- Numeric and categorical features ---
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
                    'waterfront', 'view', 'condition', 'grade', 'sqft_above',
                    'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long',
                    'sqft_living15', 'sqft_lot15', 'zip_mean']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['zipcode']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

In [4]:
# Custom transformer to add zipcode average price
class ZipAvgPriceAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.zip_avg_price = X.groupby('zipcode')['price'].mean().reset_index(name='zip_mean')
        return self
    
    def transform(self, X, y=None):
        return X.merge(self.zip_avg_price, on='zipcode', how='left')

In [5]:
# King County House Sales dataset from OpenML (includes Seattle)
# this is an ARFF file, which is a text file with a specific format
# --- Load dataset (Pandas) ---
url = 'https://www.openml.org/data/download/22044765/dataset'
cols = ['id', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
        'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement',
        'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15',
        'sqft_lot15', 'date_year', 'date_month', 'date_day']

raw = pd.read_csv(url, names=cols, skiprows=31)

In [6]:
y = raw['price']
X_train, X_test, y_train, y_test = train_test_split(raw, y, test_size=0.2, random_state=42)

lr = LinearRegression()
tweak_transformer = FunctionTransformer(tweak_housing)

lr_pipe = Pipeline(steps=[
    ('tweak', tweak_transformer),
    ('zip_avg_price', ZipAvgPriceAdder()),
    ('preprocessor', preprocessor),
    ('lr', lr)
])

lr_pipe.fit(X_train, y_train)
print(lr_pipe.score(X_test, y_test))

0.8069700278216327
